# 11 - Deep Learning Model 3: FT-Transformer (PyTorch)
Each categorical feature becomes one token, a [CLS] token is prepended, and self-attention learns feature interactions.
Uses the same leakage-free split, validation-only early stopping and results table as the other models.

In [1]:
# Make sure we run from the project root (notebook lives in /notebooks) so data_utils imports
import os, sys
if not os.path.exists("data_utils.py"):
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Working dir:", os.getcwd())

Working dir: c:\Users\aru12\OneDrive\Desktop\capstone project


In [2]:
import copy, numpy as np, pandas as pd, torch, torch.nn as nn
from sklearn.metrics import roc_auc_score
from data_utils import *

torch.manual_seed(SEED); np.random.seed(SEED)
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", dev)
(Xtr, ytr), (Xva, yva), (Xte, yte) = load_splits()
print(len(Xtr), len(Xva), len(Xte))

Device: cpu
389196 131775 236927


## Encode categories (vocabularies from TRAIN only; unseen -> index 0)

In [3]:
vocab = {c: {v: i + 1 for i, v in enumerate(sorted(Xtr[c].unique()))} for c in FEATURES}
def enc(X):
    return torch.tensor(np.stack([X[c].map(vocab[c]).fillna(0).astype(int).to_numpy()
                                  for c in FEATURES], 1), dtype=torch.long).to(dev)
Ctr, Cva, Cte = enc(Xtr), enc(Xva), enc(Xte)
Ytr = torch.tensor(ytr, dtype=torch.float32).to(dev)

## Model

In [4]:
class FTTransformer(nn.Module):
    """Each categorical feature -> one token; a [CLS] token is prepended; output is read from [CLS]."""
    def __init__(self, d=64, heads=4, layers=3, ff=128, p=0.1):
        super().__init__()
        sizes = [len(vocab[c]) + 1 for c in FEATURES]
        self.register_buffer("offsets", torch.tensor([0] + sizes[:-1]).cumsum(0))
        self.emb = nn.Embedding(sum(sizes), d)                          # one shared table, per-feature offsets
        self.feat_bias = nn.Parameter(torch.zeros(len(FEATURES), d))    # per-feature bias
        self.cls = nn.Parameter(torch.randn(1, 1, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, heads, ff, dropout=p, activation="gelu",
                                           batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.head = nn.Sequential(nn.LayerNorm(d), nn.ReLU(), nn.Linear(d, 1))
    def forward(self, c):
        x = self.emb(c + self.offsets) + self.feat_bias                 # (B, F, d)
        x = torch.cat([self.cls.expand(len(x), -1, -1), x], 1)          # (B, F+1, d)
        return self.head(self.enc(x)[:, 0]).squeeze(1)

net = FTTransformer().to(dev)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor((ytr == 0).sum() / (ytr == 1).sum(),
                                                        dtype=torch.float32).to(dev))
opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=1)

def predict(C):
    net.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(C), 8192):
            out.append(torch.sigmoid(net(C[i:i + 8192])))
    return torch.cat(out).cpu().numpy()

## Train (early stopping + LR schedule use VALIDATION only)
If it is too slow on CPU, set `BS = 4096`.

In [5]:
best_auc, best_state, bad, BS = 0, None, 0, 1024
for ep in range(30):
    net.train(); perm = torch.randperm(len(Ctr), device=dev)
    for i in range(0, len(perm), BS):
        idx = perm[i:i + BS]
        opt.zero_grad()
        loss_fn(net(Ctr[idx]), Ytr[idx]).backward()
        nn.utils.clip_grad_norm_(net.parameters(), 1.0)
        opt.step()
    auc = roc_auc_score(yva, predict(Cva))
    sched.step(auc)
    print(f"epoch {ep+1:02d}  val_auc={auc:.4f}")
    if auc > best_auc: best_auc, best_state, bad = auc, copy.deepcopy(net.state_dict()), 0
    else:
        bad += 1
        if bad >= 5: break

epoch 01  val_auc=0.9139
epoch 02  val_auc=0.9161
epoch 03  val_auc=0.9177
epoch 04  val_auc=0.9181
epoch 05  val_auc=0.9180
epoch 06  val_auc=0.9179
epoch 07  val_auc=0.9185
epoch 08  val_auc=0.9184
epoch 09  val_auc=0.9182
epoch 10  val_auc=0.9185
epoch 11  val_auc=0.9180
epoch 12  val_auc=0.9181


## Evaluate on the TEST set and save results

In [6]:
net.load_state_dict(best_state)
t = best_threshold(yva, predict(Cva))                 # threshold chosen on validation
p_te = predict(Cte)
report("FT-Transformer (DL)", yte, p_te, t)
print("test ROC-AUC:", round(roc_auc_score(yte, p_te), 4))
save_rows([metrics_row("FT-Transformer (DL)", yte, p_te, t)])
print("Saved FT-Transformer row to", RESULTS)

FT-Transformer (DL)    thr=0.53  accuracy=0.8321  balanced_acc=0.8339  gap=0.0017
test ROC-AUC: 0.9137
Saved FT-Transformer row to c:\Users\aru12\OneDrive\Desktop\capstone project\outputs\leakfree_results.csv


## Refresh the comparison chart

In [7]:
!python evaluate_ml.py


FINAL RESULTS (test set = all of 2025)

               Model  Test_rows  Threshold  Accuracy  Balanced_Accuracy  Precision  Recall     F1  ROC_AUC
            LightGBM     236927       0.55    0.8302             0.8301     0.5353  0.8299 0.6508   0.9114
    MLP (embeddings)     236927       0.57    0.8327             0.8336     0.5397  0.8350 0.6556   0.9136
 FT-Transformer (DL)     236927       0.53    0.8321             0.8339     0.5385  0.8367 0.6552   0.9137
  LogisticRegression     236927       0.53    0.8271             0.8276     0.5299  0.8285 0.6463   0.9094
        RandomForest     236927       0.51    0.8304             0.8301     0.5357  0.8295 0.6510   0.9104
HistGradientBoosting     236927       0.55    0.8296             0.8299     0.5342  0.8303 0.6501   0.9113
             XGBoost     236927       0.52    0.8312             0.8311     0.5371  0.8309 0.6524   0.9125

Features used by every model: Primary Type + Location Description + Beat
Saved: c:\Users\aru12\OneDriv